# Goal

Explore chart run duration and gaps between chart runs to understand
how songs leave and return to the UK Singles Chart.

The analyses focus on weeks on chart, recurring chart runs, and gaps
between runs to identify patterns and potential data quality issues.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Load Datasets

project_path = Path.cwd().parent
interim_path = project_path / 'data' / 'interim'

chart_history = pd.read_csv(interim_path/'uk_chart_history_clean.csv')
songs = pd.read_csv(interim_path/'songs.csv')


In [3]:
chart_history.info()

<class 'pandas.DataFrame'>
RangeIndex: 210882 entries, 0 to 210881
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   Song            210882 non-null  str  
 1   Artist          210882 non-null  str  
 2   Position        210882 non-null  int64
 3   Last Week       210882 non-null  str  
 4   Peak            210882 non-null  int64
 5   Weeks on Chart  210882 non-null  int64
 6   Week            210882 non-null  str  
dtypes: int64(3), str(4)
memory usage: 11.3 MB


In [4]:
songs.info()

<class 'pandas.DataFrame'>
RangeIndex: 36652 entries, 0 to 36651
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   song_id  36652 non-null  int64
 1   Song     36652 non-null  str  
 2   Artist   36652 non-null  str  
dtypes: int64(1), str(2)
memory usage: 859.2 KB


### Link Song Identifiers

Song identifiers are merged into the chart history to establish a stable relationship
between chart entries and song-based analyses.

In [5]:
chart_history = (chart_history.merge(songs,
                                     on=['Song', 'Artist'],
                                     how='left'))

chart_history = (chart_history[['song_id',
                                'Song',
                                'Artist',
                                'Position',
                                'Last Week',
                                'Peak',
                                'Weeks on Chart',
                                'Week']])

chart_history.head()

,song_id,Song,Artist,Position,Last Week,Peak,Weeks on Chart,Week
0,1,SAVE YOUR LOVE,RENEE AND RENATO,1,LW:1,1,11,2 January 1983- 8 January 1983
1,2,YOU CAN'T HURRY LOVE,PHIL COLLINS,2,LW:6,2,6,2 January 1983- 8 January 1983
2,3,A WINTER'S TALE,DAVID ESSEX,3,LW:7,3,5,2 January 1983- 8 January 1983
3,4,BEST YEARS OF OUR LIVES,MODERN ROMANCE,4,LW:8,4,9,2 January 1983- 8 January 1983
4,5,OUR HOUSE,MADNESS,5,LW:5,5,7,2 January 1983- 8 January 1983


## Explore Chart Duration

Chart duration is defined as the number of weeks a song remains in the UK Top 100.

The following analyses examine the distribution of chart longevity and explore
whether chart duration has changed over time.

In [6]:
chart_history['Weeks on Chart'].describe()

count    210882.000000
mean         10.096713
std          17.379638
min           1.000000
25%           2.000000
50%           5.000000
75%          11.000000
max         403.000000
Name: Weeks on Chart, dtype: float64

In [7]:
chart_history['Weeks on Chart'].value_counts().sort_index().head()

Weeks on Chart
1    36401
2    27965
3    19475
4    15527
5    12841
Name: count, dtype: int64

#### How does Week on Charts count?

In [8]:
print(f'Weeks on chart maximum: {chart_history['Weeks on Chart'].max()}')

Weeks on chart maximum: 403


In [9]:
# Identify the song with the longest cumulative chart duration

weeks_on_charts_max = (chart_history['Weeks on Chart'] == chart_history['Weeks on Chart'].max())

chart_history.loc[weeks_on_charts_max]

,song_id,Song,Artist,Position,Last Week,Peak,Weeks on Chart,Week
210847,24702,MR BRIGHTSIDE,KILLERS,66,LW:65,10,403,29 March 2024- 4 April 2024


In [10]:
mr_brightside = chart_history['song_id'] == 24702.0

chart_history.loc[mr_brightside, :].shape

(402, 8)

In [11]:
chart_history.head()

,song_id,Song,Artist,Position,Last Week,Peak,Weeks on Chart,Week
0,1,SAVE YOUR LOVE,RENEE AND RENATO,1,LW:1,1,11,2 January 1983- 8 January 1983
1,2,YOU CAN'T HURRY LOVE,PHIL COLLINS,2,LW:6,2,6,2 January 1983- 8 January 1983
2,3,A WINTER'S TALE,DAVID ESSEX,3,LW:7,3,5,2 January 1983- 8 January 1983
3,4,BEST YEARS OF OUR LIVES,MODERN ROMANCE,4,LW:8,4,9,2 January 1983- 8 January 1983
4,5,OUR HOUSE,MADNESS,5,LW:5,5,7,2 January 1983- 8 January 1983


## Use OUR HOUSE as an example dataset

OUR HOUSE is used as a small, representative example to illustrate
how the chart data records cumulative chart weeks and re-entries.

The song's complete chart history is selected using its song_id and
will be used throughout the following steps to demonstrate the
calculation of individual chart runs.

In [12]:
madness = chart_history['song_id'] == 5.0

chart_history.loc[madness, :]

,song_id,Song,Artist,Position,Last Week,Peak,Weeks on Chart,Week
4,5,OUR HOUSE,MADNESS,5,LW:5,5,7,2 January 1983- 8 January 1983
112,5,OUR HOUSE,MADNESS,13,LW:5,5,8,9 January 1983- 15 January 1983
208,5,OUR HOUSE,MADNESS,10,LW:13,5,9,16 January 1983- 22 January 1983
319,5,OUR HOUSE,MADNESS,21,LW:10,5,10,23 January 1983- 29 January 1983
433,5,OUR HOUSE,MADNESS,35,LW:21,5,11,30 January 1983- 5 February 1983
551,5,OUR HOUSE,MADNESS,53,LW:35,5,12,6 February 1983- 12 February 1983
671,5,OUR HOUSE,MADNESS,73,LW:53,5,13,13 February 1983- 19 February 1983
149375,5,OUR HOUSE,MADNESS,92,LW:RE,5,14,10 June 2012- 16 June 2012
150382,5,OUR HOUSE,MADNESS,99,LW:RE,5,15,19 August 2012- 25 August 2012


In [13]:
len(chart_history.loc[madness, :])

9

### Validation:

- `Weeks on Chart` counts the cumulative number of weeks a song has spent in the charts.
- Only weeks in which the song appears in the UK Top 100 are counted; time outside the chart is not included.
- Re-entry weeks continue the existing count rather than restarting it.

## Identify individual chart runs

`Weeks on Chart` counts the cumulative number of weeks a song has spent
in the charts. To analyse the duration of individual chart runs,
the chart history is copied and separate chart runs are identified.

In [14]:
chart_duration = chart_history.copy()
chart_duration.head()

,song_id,Song,Artist,Position,Last Week,Peak,Weeks on Chart,Week
0,1,SAVE YOUR LOVE,RENEE AND RENATO,1,LW:1,1,11,2 January 1983- 8 January 1983
1,2,YOU CAN'T HURRY LOVE,PHIL COLLINS,2,LW:6,2,6,2 January 1983- 8 January 1983
2,3,A WINTER'S TALE,DAVID ESSEX,3,LW:7,3,5,2 January 1983- 8 January 1983
3,4,BEST YEARS OF OUR LIVES,MODERN ROMANCE,4,LW:8,4,9,2 January 1983- 8 January 1983
4,5,OUR HOUSE,MADNESS,5,LW:5,5,7,2 January 1983- 8 January 1983


In [15]:
# Sort chart history chronologically by song and week

chart_duration['_Week Start'] = pd.to_datetime(chart_duration['Week'].str.split('-').str[0], format='%d %B %Y')

chart_duration = (chart_duration.sort_values(['song_id', '_Week Start']))

chart_duration.loc[madness, :]

,song_id,Song,Artist,Position,Last Week,Peak,Weeks on Chart,Week,_Week Start
4,5,OUR HOUSE,MADNESS,5,LW:5,5,7,2 January 1983- 8 January 1983,1983-01-02
112,5,OUR HOUSE,MADNESS,13,LW:5,5,8,9 January 1983- 15 January 1983,1983-01-09
208,5,OUR HOUSE,MADNESS,10,LW:13,5,9,16 January 1983- 22 January 1983,1983-01-16
319,5,OUR HOUSE,MADNESS,21,LW:10,5,10,23 January 1983- 29 January 1983,1983-01-23
433,5,OUR HOUSE,MADNESS,35,LW:21,5,11,30 January 1983- 5 February 1983,1983-01-30
551,5,OUR HOUSE,MADNESS,53,LW:35,5,12,6 February 1983- 12 February 1983,1983-02-06
671,5,OUR HOUSE,MADNESS,73,LW:53,5,13,13 February 1983- 19 February 1983,1983-02-13
149375,5,OUR HOUSE,MADNESS,92,LW:RE,5,14,10 June 2012- 16 June 2012,2012-06-10
150382,5,OUR HOUSE,MADNESS,99,LW:RE,5,15,19 August 2012- 25 August 2012,2012-08-19


In [16]:
# Drop unused columns

chart_duration = chart_duration.drop(columns=['Peak'])
chart_duration.loc[madness, :]

,song_id,Song,Artist,Position,Last Week,Weeks on Chart,Week,_Week Start
4,5,OUR HOUSE,MADNESS,5,LW:5,7,2 January 1983- 8 January 1983,1983-01-02
112,5,OUR HOUSE,MADNESS,13,LW:5,8,9 January 1983- 15 January 1983,1983-01-09
208,5,OUR HOUSE,MADNESS,10,LW:13,9,16 January 1983- 22 January 1983,1983-01-16
319,5,OUR HOUSE,MADNESS,21,LW:10,10,23 January 1983- 29 January 1983,1983-01-23
433,5,OUR HOUSE,MADNESS,35,LW:21,11,30 January 1983- 5 February 1983,1983-01-30
551,5,OUR HOUSE,MADNESS,53,LW:35,12,6 February 1983- 12 February 1983,1983-02-06
671,5,OUR HOUSE,MADNESS,73,LW:53,13,13 February 1983- 19 February 1983,1983-02-13
149375,5,OUR HOUSE,MADNESS,92,LW:RE,14,10 June 2012- 16 June 2012,2012-06-10
150382,5,OUR HOUSE,MADNESS,99,LW:RE,15,19 August 2012- 25 August 2012,2012-08-19


### Identify chart entries without a new entry or re-entry marker

Not every chart entry has an explicit `NEW` or `RE` marker. 
These entries need to be identified before defining individual chart runs.

In [17]:
# Explore new entries and re-entries

songs_new_re = (chart_duration.loc[:, 'Last Week'] == 'LW:New') | (
                chart_duration.loc[:, 'Last Week'] == 'LW:RE')

chart_duration.loc[songs_new_re, :].head()

,song_id,Song,Artist,Position,Last Week,Weeks on Chart,Week,_Week Start
149375,5,OUR HOUSE,MADNESS,92,LW:RE,14,10 June 2012- 16 June 2012,2012-06-10
150382,5,OUR HOUSE,MADNESS,99,LW:RE,15,19 August 2012- 25 August 2012,2012-08-19
1492,6,TIME (CLOCK OF THE HEART),CULTURE CLUB,94,LW:RE,13,10 April 1983- 16 April 1983,1983-04-10
37,38,DOWN UNDER,MEN AT WORK,38,LW:New,1,2 January 1983- 8 January 1983,1983-01-02
39,40,EUROPEAN FEMALE,THE STRANGLERS,40,LW:New,1,2 January 1983- 8 January 1983,1983-01-02


In [18]:
# Identify new entries and re-entries

chart_duration.loc[songs_new_re, 'Last Week'].value_counts()

Last Week
LW:New    36401
LW:RE     10297
Name: count, dtype: int64

In [19]:
# Count entries without an explicit new entry or re-entry marker

print(f'Song entries without a New or RE entry marker: {len(chart_duration.loc[~songs_new_re, :])}')

Song entries without a New or RE entry marker: 164184


### Number the chart entries for each song

To identify the first observed chart entry of each song, the chart history is numbered sequentially within each `song_id`.

In [20]:
chart_duration['song_entry'] = (chart_duration
                                .groupby('song_id')
                                .cumcount())

chart_duration.loc[madness,:]

,song_id,Song,Artist,Position,Last Week,Weeks on Chart,Week,_Week Start,song_entry
4,5,OUR HOUSE,MADNESS,5,LW:5,7,2 January 1983- 8 January 1983,1983-01-02,0
112,5,OUR HOUSE,MADNESS,13,LW:5,8,9 January 1983- 15 January 1983,1983-01-09,1
208,5,OUR HOUSE,MADNESS,10,LW:13,9,16 January 1983- 22 January 1983,1983-01-16,2
319,5,OUR HOUSE,MADNESS,21,LW:10,10,23 January 1983- 29 January 1983,1983-01-23,3
433,5,OUR HOUSE,MADNESS,35,LW:21,11,30 January 1983- 5 February 1983,1983-01-30,4
551,5,OUR HOUSE,MADNESS,53,LW:35,12,6 February 1983- 12 February 1983,1983-02-06,5
671,5,OUR HOUSE,MADNESS,73,LW:53,13,13 February 1983- 19 February 1983,1983-02-13,6
149375,5,OUR HOUSE,MADNESS,92,LW:RE,14,10 June 2012- 16 June 2012,2012-06-10,7
150382,5,OUR HOUSE,MADNESS,99,LW:RE,15,19 August 2012- 25 August 2012,2012-08-19,8


### Identify chart run starts

To analyse the duration of individual chart runs, entries that mark
the start of a chart run are classified as `NEW`, `RE`, or `OLD`.

In [21]:
chart_duration['Run Start'] = ''

new_week = chart_duration['Last Week'] == 'LW:New'
chart_duration.loc[new_week, 'Run Start'] = 'NEW'

re_week = chart_duration['Last Week'] == 'LW:RE'
chart_duration.loc[re_week, 'Run Start'] = 'RE'

first_entry = chart_duration['song_entry'] == 0
old_week = first_entry & ~songs_new_re

chart_duration.loc[old_week, 'Run Start'] = 'OLD'

chart_duration.loc[madness,:]

,song_id,Song,Artist,Position,Last Week,Weeks on Chart,Week,_Week Start,song_entry,Run Start
4,5,OUR HOUSE,MADNESS,5,LW:5,7,2 January 1983- 8 January 1983,1983-01-02,0,OLD
112,5,OUR HOUSE,MADNESS,13,LW:5,8,9 January 1983- 15 January 1983,1983-01-09,1,
208,5,OUR HOUSE,MADNESS,10,LW:13,9,16 January 1983- 22 January 1983,1983-01-16,2,
319,5,OUR HOUSE,MADNESS,21,LW:10,10,23 January 1983- 29 January 1983,1983-01-23,3,
433,5,OUR HOUSE,MADNESS,35,LW:21,11,30 January 1983- 5 February 1983,1983-01-30,4,
551,5,OUR HOUSE,MADNESS,53,LW:35,12,6 February 1983- 12 February 1983,1983-02-06,5,
671,5,OUR HOUSE,MADNESS,73,LW:53,13,13 February 1983- 19 February 1983,1983-02-13,6,
149375,5,OUR HOUSE,MADNESS,92,LW:RE,14,10 June 2012- 16 June 2012,2012-06-10,7,RE
150382,5,OUR HOUSE,MADNESS,99,LW:RE,15,19 August 2012- 25 August 2012,2012-08-19,8,RE


In [22]:
chart_duration['Run Start'].value_counts()

Run Start
       163826
NEW     36401
RE      10297
OLD       358
Name: count, dtype: int64

In [23]:
chart_duration.loc[old_week, :].head()

,song_id,Song,Artist,Position,Last Week,Weeks on Chart,Week,_Week Start,song_entry,Run Start
0,1,SAVE YOUR LOVE,RENEE AND RENATO,1,LW:1,11,2 January 1983- 8 January 1983,1983-01-02,0,OLD
1,2,YOU CAN'T HURRY LOVE,PHIL COLLINS,2,LW:6,6,2 January 1983- 8 January 1983,1983-01-02,0,OLD
2,3,A WINTER'S TALE,DAVID ESSEX,3,LW:7,5,2 January 1983- 8 January 1983,1983-01-02,0,OLD
3,4,BEST YEARS OF OUR LIVES,MODERN ROMANCE,4,LW:8,9,2 January 1983- 8 January 1983,1983-01-02,0,OLD
4,5,OUR HOUSE,MADNESS,5,LW:5,7,2 January 1983- 8 January 1983,1983-01-02,0,OLD


### Create a unique ID for each chart run

A new chart run starts whenever `Run Start` contains `NEW`, `RE`, or `OLD`.

A cumulative count of these run starts is used to assign a unique `Run ID` to each chart run within each song.

In [24]:
run_start = chart_duration['Run Start'] != ''

chart_duration['Run ID'] = (run_start.groupby(chart_duration['song_id']).cumsum())
chart_duration.loc[madness,:]

,song_id,Song,Artist,Position,Last Week,Weeks on Chart,Week,_Week Start,song_entry,Run Start,Run ID
4,5,OUR HOUSE,MADNESS,5,LW:5,7,2 January 1983- 8 January 1983,1983-01-02,0,OLD,1
112,5,OUR HOUSE,MADNESS,13,LW:5,8,9 January 1983- 15 January 1983,1983-01-09,1,,1
208,5,OUR HOUSE,MADNESS,10,LW:13,9,16 January 1983- 22 January 1983,1983-01-16,2,,1
319,5,OUR HOUSE,MADNESS,21,LW:10,10,23 January 1983- 29 January 1983,1983-01-23,3,,1
433,5,OUR HOUSE,MADNESS,35,LW:21,11,30 January 1983- 5 February 1983,1983-01-30,4,,1
551,5,OUR HOUSE,MADNESS,53,LW:35,12,6 February 1983- 12 February 1983,1983-02-06,5,,1
671,5,OUR HOUSE,MADNESS,73,LW:53,13,13 February 1983- 19 February 1983,1983-02-13,6,,1
149375,5,OUR HOUSE,MADNESS,92,LW:RE,14,10 June 2012- 16 June 2012,2012-06-10,7,RE,2
150382,5,OUR HOUSE,MADNESS,99,LW:RE,15,19 August 2012- 25 August 2012,2012-08-19,8,RE,3


### Calculate the duration of each chart run

The number of weeks belonging to each individual chart run is calculated
by counting the rows for each combination of `song_id` and `Run ID`.

The resulting values are stored temporarily in `run_count` and used
to assign the duration to the corresponding run-start rows.

In [25]:
# Count the number of weeks in each individual chart run

run_count = chart_duration.groupby(['song_id', 'Run ID'])['Run ID'].transform('count')

run_count[madness]

4         7
112       7
208       7
319       7
433       7
551       7
671       7
149375    1
150382    1
Name: Run ID, dtype: int64

In [26]:
# Store the run duration for new entries and re-entries

new_re = chart_duration['Run Start'].isin(['NEW', 'RE'])

chart_duration.loc[new_re, 'Run Count'] = run_count[new_re]

chart_duration.loc[madness, :]

,song_id,Song,Artist,Position,Last Week,Weeks on Chart,Week,_Week Start,song_entry,Run Start,Run ID,Run Count
4,5,OUR HOUSE,MADNESS,5,LW:5,7,2 January 1983- 8 January 1983,1983-01-02,0,OLD,1,NaN
112,5,OUR HOUSE,MADNESS,13,LW:5,8,9 January 1983- 15 January 1983,1983-01-09,1,,1,NaN
208,5,OUR HOUSE,MADNESS,10,LW:13,9,16 January 1983- 22 January 1983,1983-01-16,2,,1,NaN
319,5,OUR HOUSE,MADNESS,21,LW:10,10,23 January 1983- 29 January 1983,1983-01-23,3,,1,NaN
433,5,OUR HOUSE,MADNESS,35,LW:21,11,30 January 1983- 5 February 1983,1983-01-30,4,,1,NaN
551,5,OUR HOUSE,MADNESS,53,LW:35,12,6 February 1983- 12 February 1983,1983-02-06,5,,1,NaN
671,5,OUR HOUSE,MADNESS,73,LW:53,13,13 February 1983- 19 February 1983,1983-02-13,6,,1,NaN
149375,5,OUR HOUSE,MADNESS,92,LW:RE,14,10 June 2012- 16 June 2012,2012-06-10,7,RE,2,1.0
150382,5,OUR HOUSE,MADNESS,99,LW:RE,15,19 August 2012- 25 August 2012,2012-08-19,8,RE,3,1.0


#### Store the duration of new entries and re-entries

For `NEW` and `RE` runs, the calculated number of weeks can be assigned directly to the corresponding run-start row.

In [27]:
# Store the run duration for runs already in progress at the start of the dataset

old = chart_duration['Run Start'] == 'OLD'

chart_duration.loc[old, 'Run Count'] = run_count[old]

chart_duration.loc[madness, :]


,song_id,Song,Artist,Position,Last Week,Weeks on Chart,Week,_Week Start,song_entry,Run Start,Run ID,Run Count
4,5,OUR HOUSE,MADNESS,5,LW:5,7,2 January 1983- 8 January 1983,1983-01-02,0,OLD,1,7.0
112,5,OUR HOUSE,MADNESS,13,LW:5,8,9 January 1983- 15 January 1983,1983-01-09,1,,1,NaN
208,5,OUR HOUSE,MADNESS,10,LW:13,9,16 January 1983- 22 January 1983,1983-01-16,2,,1,NaN
319,5,OUR HOUSE,MADNESS,21,LW:10,10,23 January 1983- 29 January 1983,1983-01-23,3,,1,NaN
433,5,OUR HOUSE,MADNESS,35,LW:21,11,30 January 1983- 5 February 1983,1983-01-30,4,,1,NaN
551,5,OUR HOUSE,MADNESS,53,LW:35,12,6 February 1983- 12 February 1983,1983-02-06,5,,1,NaN
671,5,OUR HOUSE,MADNESS,73,LW:53,13,13 February 1983- 19 February 1983,1983-02-13,6,,1,NaN
149375,5,OUR HOUSE,MADNESS,92,LW:RE,14,10 June 2012- 16 June 2012,2012-06-10,7,RE,2,1.0
150382,5,OUR HOUSE,MADNESS,99,LW:RE,15,19 August 2012- 25 August 2012,2012-08-19,8,RE,3,1.0


#### Calculate the complete duration of an OLD chart run

For songs already present at the beginning of the dataset, `Weeks on Chart`
includes weeks accumulated before the observed chart history starts.

The complete duration of the current run is therefore calculated by
combining these previously accumulated weeks with the weeks observed
in the current run.

The resulting values are calculated separately first and will be added
back to `chart_duration` afterwards.

In [28]:
# Add the cumulative weeks already spent in the charts
# to the weeks counted in the current OLD run.

old_run_count = (chart_duration.loc[old, 'Weeks on Chart']
                 + chart_duration.loc[old, 'Run Count'])
old_run_count.head()

0    17.0
1    17.0
2    11.0
3    14.0
4    14.0
dtype: float64

In [29]:
# Create a lookup table for the calculated OLD run durations

old_values = chart_duration.loc[old, ['song_id', 'Run ID']].copy()

old_values['OLD Run Count'] = old_run_count

old_values.head()

,song_id,Run ID,OLD Run Count
0,1,1,17.0
1,2,1,17.0
2,3,1,11.0
3,4,1,14.0
4,5,1,14.0


### Create a key for returning calculated values to the chart history

The calculated `OLD Run Count` values are linked to each individual chart run
using a unique combination of `song_id` and `Run ID`.

The same key is created in `chart_duration` so that the calculated values can
later be mapped back to the corresponding chart runs.

In [30]:
# Create a unique key for each song and chart run

old_values['Run Key'] = (old_values['song_id'].astype(str)
    + '_'
    + old_values['Run ID'].astype(str))
old_values.head()

,song_id,Run ID,OLD Run Count,Run Key
0,1,1,17.0,1_1
1,2,1,17.0,2_1
2,3,1,11.0,3_1
3,4,1,14.0,4_1
4,5,1,14.0,5_1


In [31]:
# Create the same key in the chart history

chart_duration['Run Key'] = (chart_duration['song_id'].astype(str)
                             + '_'
                             + chart_duration['Run ID'].astype(str))
chart_duration.loc[madness, :]

,song_id,Song,Artist,Position,Last Week,Weeks on Chart,Week,_Week Start,song_entry,Run Start,Run ID,Run Count,Run Key
4,5,OUR HOUSE,MADNESS,5,LW:5,7,2 January 1983- 8 January 1983,1983-01-02,0,OLD,1,7.0,5_1
112,5,OUR HOUSE,MADNESS,13,LW:5,8,9 January 1983- 15 January 1983,1983-01-09,1,,1,NaN,5_1
208,5,OUR HOUSE,MADNESS,10,LW:13,9,16 January 1983- 22 January 1983,1983-01-16,2,,1,NaN,5_1
319,5,OUR HOUSE,MADNESS,21,LW:10,10,23 January 1983- 29 January 1983,1983-01-23,3,,1,NaN,5_1
433,5,OUR HOUSE,MADNESS,35,LW:21,11,30 January 1983- 5 February 1983,1983-01-30,4,,1,NaN,5_1
551,5,OUR HOUSE,MADNESS,53,LW:35,12,6 February 1983- 12 February 1983,1983-02-06,5,,1,NaN,5_1
671,5,OUR HOUSE,MADNESS,73,LW:53,13,13 February 1983- 19 February 1983,1983-02-13,6,,1,NaN,5_1
149375,5,OUR HOUSE,MADNESS,92,LW:RE,14,10 June 2012- 16 June 2012,2012-06-10,7,RE,2,1.0,5_2
150382,5,OUR HOUSE,MADNESS,99,LW:RE,15,19 August 2012- 25 August 2012,2012-08-19,8,RE,3,1.0,5_3


### Map the calculated OLD run durations back to the chart history

The complete duration of each `OLD` chart run has now been calculated
separately in `old_values`.

Using `Run Key` as the link between the helper table and `chart_duration`,
these values are mapped back to every row belonging to the corresponding
chart run.

In [32]:
# Create a mapping from each Run Key to its calculated OLD run duration

old_run_mapping = old_values.set_index('Run Key')['OLD Run Count']
old_run_mapping.head()

Run Key
1_1    17.0
2_1    17.0
3_1    11.0
4_1    14.0
5_1    14.0
Name: OLD Run Count, dtype: float64

In [33]:
print(f"Value for MADNESS OLD: {old_run_mapping.loc['5_1']}")

Value for MADNESS OLD: 14.0


In [34]:
# Map the calculated OLD run duration back to all rows of the corresponding run

old_run_values = chart_duration['Run Key'].map(old_run_mapping)
old_run_values[madness]

4         14.0
112       14.0
208       14.0
319       14.0
433       14.0
551       14.0
671       14.0
149375     NaN
150382     NaN
Name: Run Key, dtype: float64

In [35]:
# Store the calculated OLD run duration for all rows of the OLD run

chart_duration.loc[old, 'Run Count'] = old_run_values[old]
chart_duration.loc[madness, :]

,song_id,Song,Artist,Position,Last Week,Weeks on Chart,Week,_Week Start,song_entry,Run Start,Run ID,Run Count,Run Key
4,5,OUR HOUSE,MADNESS,5,LW:5,7,2 January 1983- 8 January 1983,1983-01-02,0,OLD,1,14.0,5_1
112,5,OUR HOUSE,MADNESS,13,LW:5,8,9 January 1983- 15 January 1983,1983-01-09,1,,1,NaN,5_1
208,5,OUR HOUSE,MADNESS,10,LW:13,9,16 January 1983- 22 January 1983,1983-01-16,2,,1,NaN,5_1
319,5,OUR HOUSE,MADNESS,21,LW:10,10,23 January 1983- 29 January 1983,1983-01-23,3,,1,NaN,5_1
433,5,OUR HOUSE,MADNESS,35,LW:21,11,30 January 1983- 5 February 1983,1983-01-30,4,,1,NaN,5_1
551,5,OUR HOUSE,MADNESS,53,LW:35,12,6 February 1983- 12 February 1983,1983-02-06,5,,1,NaN,5_1
671,5,OUR HOUSE,MADNESS,73,LW:53,13,13 February 1983- 19 February 1983,1983-02-13,6,,1,NaN,5_1
149375,5,OUR HOUSE,MADNESS,92,LW:RE,14,10 June 2012- 16 June 2012,2012-06-10,7,RE,2,1.0,5_2
150382,5,OUR HOUSE,MADNESS,99,LW:RE,15,19 August 2012- 25 August 2012,2012-08-19,8,RE,3,1.0,5_3


#### Validate the calculated run durations

The calculated run durations are checked using a second artist 
to verify that the run identification and duration calculation
work consistently across different songs and chart histories.

In [36]:
chart_duration[chart_duration['Artist'] == 'ABBA']

,song_id,Song,Artist,Position,Last Week,Weeks on Chart,Week,_Week Start,song_entry,Run Start,Run ID,Run Count,Run Key
25,26,UNDER ATTACK,ABBA,26,LW:26,5,2 January 1983- 8 January 1983,1983-01-02,0,OLD,1,9.0,26_1
125,26,UNDER ATTACK,ABBA,26,LW:26,6,9 January 1983- 15 January 1983,1983-01-09,1,,1,NaN,26_1
240,26,UNDER ATTACK,ABBA,42,LW:26,7,16 January 1983- 22 January 1983,1983-01-16,2,,1,NaN,26_1
371,26,UNDER ATTACK,ABBA,73,LW:42,8,23 January 1983- 29 January 1983,1983-01-23,3,,1,NaN,26_1
4462,907,THANK YOU FOR THE MUSIC,ABBA,64,LW:New,1,6 November 1983- 12 November 1983,1983-11-06,0,NEW,1,6.0,907_1
4541,907,THANK YOU FOR THE MUSIC,ABBA,43,LW:64,2,13 November 1983- 19 November 1983,1983-11-13,1,,1,NaN,907_1
4631,907,THANK YOU FOR THE MUSIC,ABBA,33,LW:43,3,20 November 1983- 26 November 1983,1983-11-20,2,,1,NaN,907_1
4732,907,THANK YOU FOR THE MUSIC,ABBA,34,LW:33,4,27 November 1983- 3 December 1983,1983-11-27,3,,1,NaN,907_1
4841,907,THANK YOU FOR THE MUSIC,ABBA,43,LW:34,5,4 December 1983- 10 December 1983,1983-12-04,4,,1,NaN,907_1
4955,907,THANK YOU FOR THE MUSIC,ABBA,57,LW:43,6,11 December 1983- 17 December 1983,1983-12-11,5,,1,NaN,907_1


## Prepare for Gap Analysis

The chart history has now been reduced to individual chart runs.

The following steps prepare the run-level dataset for gap analysis:

- add run end dates
- reduce chart history to run-start rows
- calculate run durations
- add next run start dates
- calculate gap durations in weeks
- validate negative gaps and identify overlapping runs

In [37]:
chart_duration.info()

<class 'pandas.DataFrame'>
Index: 210882 entries, 0 to 210879
Data columns (total 13 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   song_id         210882 non-null  int64         
 1   Song            210882 non-null  str           
 2   Artist          210882 non-null  str           
 3   Position        210882 non-null  int64         
 4   Last Week       210882 non-null  str           
 5   Weeks on Chart  210882 non-null  int64         
 6   Week            210882 non-null  str           
 7   _Week Start     210882 non-null  datetime64[us]
 8   song_entry      210882 non-null  int64         
 9   Run Start       210882 non-null  str           
 10  Run ID          210882 non-null  int64         
 11  Run Count       47056 non-null   float64       
 12  Run Key         210882 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(5), str(6)
memory usage: 22.5 MB


#### Remove redundant columns

The information from `Last Week` has already been used to classify
the start of each chart run.

`Weeks on Chart` is no longer required because the duration of each
individual run is stored in `Run Count`.

Both columns can therefore be removed before reducing the dataset
to one row per chart run.

In [38]:
# Remove columns that are no longer needed

chart_duration = chart_duration.drop(columns=['Last Week', 'Weeks on Chart'])
chart_duration.loc[madness, :]

,song_id,Song,Artist,Position,Week,_Week Start,song_entry,Run Start,Run ID,Run Count,Run Key
4,5,OUR HOUSE,MADNESS,5,2 January 1983- 8 January 1983,1983-01-02,0,OLD,1,14.0,5_1
112,5,OUR HOUSE,MADNESS,13,9 January 1983- 15 January 1983,1983-01-09,1,,1,NaN,5_1
208,5,OUR HOUSE,MADNESS,10,16 January 1983- 22 January 1983,1983-01-16,2,,1,NaN,5_1
319,5,OUR HOUSE,MADNESS,21,23 January 1983- 29 January 1983,1983-01-23,3,,1,NaN,5_1
433,5,OUR HOUSE,MADNESS,35,30 January 1983- 5 February 1983,1983-01-30,4,,1,NaN,5_1
551,5,OUR HOUSE,MADNESS,53,6 February 1983- 12 February 1983,1983-02-06,5,,1,NaN,5_1
671,5,OUR HOUSE,MADNESS,73,13 February 1983- 19 February 1983,1983-02-13,6,,1,NaN,5_1
149375,5,OUR HOUSE,MADNESS,92,10 June 2012- 16 June 2012,2012-06-10,7,RE,2,1.0,5_2
150382,5,OUR HOUSE,MADNESS,99,19 August 2012- 25 August 2012,2012-08-19,8,RE,3,1.0,5_3


In [39]:
### Extract the end date of each chart week

In [40]:
chart_duration['_Week End'] = pd.to_datetime(chart_duration['Week'].str.split('-').str[1].str.strip(),
                                             format='%d %B %Y')
chart_duration.loc[madness, :]

,song_id,Song,Artist,Position,Week,_Week Start,song_entry,Run Start,Run ID,Run Count,Run Key,_Week End
4,5,OUR HOUSE,MADNESS,5,2 January 1983- 8 January 1983,1983-01-02,0,OLD,1,14.0,5_1,1983-01-08
112,5,OUR HOUSE,MADNESS,13,9 January 1983- 15 January 1983,1983-01-09,1,,1,NaN,5_1,1983-01-15
208,5,OUR HOUSE,MADNESS,10,16 January 1983- 22 January 1983,1983-01-16,2,,1,NaN,5_1,1983-01-22
319,5,OUR HOUSE,MADNESS,21,23 January 1983- 29 January 1983,1983-01-23,3,,1,NaN,5_1,1983-01-29
433,5,OUR HOUSE,MADNESS,35,30 January 1983- 5 February 1983,1983-01-30,4,,1,NaN,5_1,1983-02-05
551,5,OUR HOUSE,MADNESS,53,6 February 1983- 12 February 1983,1983-02-06,5,,1,NaN,5_1,1983-02-12
671,5,OUR HOUSE,MADNESS,73,13 February 1983- 19 February 1983,1983-02-13,6,,1,NaN,5_1,1983-02-19
149375,5,OUR HOUSE,MADNESS,92,10 June 2012- 16 June 2012,2012-06-10,7,RE,2,1.0,5_2,2012-06-16
150382,5,OUR HOUSE,MADNESS,99,19 August 2012- 25 August 2012,2012-08-19,8,RE,3,1.0,5_3,2012-08-25


### Prepare run-end dates for gap analysis

The last week of each chart run is identified using `Run Key`.
The resulting end date will later be stored on the corresponding
run-start row.

In [41]:
# Identify the last week of each chart run

run_end = (chart_duration
           .groupby('Run Key')['_Week End']
           .transform('max'))
run_end[madness]

4        1983-02-19
112      1983-02-19
208      1983-02-19
319      1983-02-19
433      1983-02-19
551      1983-02-19
671      1983-02-19
149375   2012-06-16
150382   2012-08-25
Name: _Week End, dtype: datetime64[us]

#### Store the end date of each chart run

The end date calculated for each `Run Key` is now stored on the row where
the corresponding chart run starts.

This preserves the run-level information before intermediate weekly entries
are removed.

In [42]:
# Store the end date of each run on its run-start row

run_start = chart_duration['Run Start'] != ''

chart_duration.loc[run_start, '_Week End'] = run_end[run_start]

chart_duration.loc[madness, :]

,song_id,Song,Artist,Position,Week,_Week Start,song_entry,Run Start,Run ID,Run Count,Run Key,_Week End
4,5,OUR HOUSE,MADNESS,5,2 January 1983- 8 January 1983,1983-01-02,0,OLD,1,14.0,5_1,1983-02-19
112,5,OUR HOUSE,MADNESS,13,9 January 1983- 15 January 1983,1983-01-09,1,,1,NaN,5_1,1983-01-15
208,5,OUR HOUSE,MADNESS,10,16 January 1983- 22 January 1983,1983-01-16,2,,1,NaN,5_1,1983-01-22
319,5,OUR HOUSE,MADNESS,21,23 January 1983- 29 January 1983,1983-01-23,3,,1,NaN,5_1,1983-01-29
433,5,OUR HOUSE,MADNESS,35,30 January 1983- 5 February 1983,1983-01-30,4,,1,NaN,5_1,1983-02-05
551,5,OUR HOUSE,MADNESS,53,6 February 1983- 12 February 1983,1983-02-06,5,,1,NaN,5_1,1983-02-12
671,5,OUR HOUSE,MADNESS,73,13 February 1983- 19 February 1983,1983-02-13,6,,1,NaN,5_1,1983-02-19
149375,5,OUR HOUSE,MADNESS,92,10 June 2012- 16 June 2012,2012-06-10,7,RE,2,1.0,5_2,2012-06-16
150382,5,OUR HOUSE,MADNESS,99,19 August 2012- 25 August 2012,2012-08-19,8,RE,3,1.0,5_3,2012-08-25


## Keep only chart-run starts

Only the first row of each chart run is needed for the gap analysis.
The intermediate weekly entries are removed, leaving one row per chart run.

In [43]:
# Keep only the run-start rows

chart_duration = chart_duration.loc[run_start, :].copy()

chart_duration.loc[madness, :]

,song_id,Song,Artist,Position,Week,_Week Start,song_entry,Run Start,Run ID,Run Count,Run Key,_Week End
4,5,OUR HOUSE,MADNESS,5,2 January 1983- 8 January 1983,1983-01-02,0,OLD,1,14.0,5_1,1983-02-19
149375,5,OUR HOUSE,MADNESS,92,10 June 2012- 16 June 2012,2012-06-10,7,RE,2,1.0,5_2,2012-06-16
150382,5,OUR HOUSE,MADNESS,99,19 August 2012- 25 August 2012,2012-08-19,8,RE,3,1.0,5_3,2012-08-25


In [44]:
chart_duration.info()

<class 'pandas.DataFrame'>
Index: 47056 entries, 0 to 210879
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   song_id      47056 non-null  int64         
 1   Song         47056 non-null  str           
 2   Artist       47056 non-null  str           
 3   Position     47056 non-null  int64         
 4   Week         47056 non-null  str           
 5   _Week Start  47056 non-null  datetime64[us]
 6   song_entry   47056 non-null  int64         
 7   Run Start    47056 non-null  str           
 8   Run ID       47056 non-null  int64         
 9   Run Count    47056 non-null  float64       
 10  Run Key      47056 non-null  str           
 11  _Week End    47056 non-null  datetime64[us]
dtypes: datetime64[us](2), float64(1), int64(4), str(5)
memory usage: 4.7 MB


In [45]:
# Drop unused columns

chart_duration = chart_duration.drop(columns=['song_entry', 'Run Key'])
chart_duration.loc[madness, :]

,song_id,Song,Artist,Position,Week,_Week Start,Run Start,Run ID,Run Count,_Week End
4,5,OUR HOUSE,MADNESS,5,2 January 1983- 8 January 1983,1983-01-02,OLD,1,14.0,1983-02-19
149375,5,OUR HOUSE,MADNESS,92,10 June 2012- 16 June 2012,2012-06-10,RE,2,1.0,2012-06-16
150382,5,OUR HOUSE,MADNESS,99,19 August 2012- 25 August 2012,2012-08-19,RE,3,1.0,2012-08-25


In [46]:
chart_duration = chart_duration.rename(columns={'_Week Start': 'Run Start Date',
                                                '_Week End': 'Run End Date',
                                                'Run Count': 'Run Weeks Count'})

chart_duration.loc[madness, :]

,song_id,Song,Artist,Position,Week,Run Start Date,Run Start,Run ID,Run Weeks Count,Run End Date
4,5,OUR HOUSE,MADNESS,5,2 January 1983- 8 January 1983,1983-01-02,OLD,1,14.0,1983-02-19
149375,5,OUR HOUSE,MADNESS,92,10 June 2012- 16 June 2012,2012-06-10,RE,2,1.0,2012-06-16
150382,5,OUR HOUSE,MADNESS,99,19 August 2012- 25 August 2012,2012-08-19,RE,3,1.0,2012-08-25


In [47]:
chart_duration = chart_duration.sort_values('song_id')
chart_duration.head(10)

,song_id,Song,Artist,Position,Week,Run Start Date,Run Start,Run ID,Run Weeks Count,Run End Date
0,1,SAVE YOUR LOVE,RENEE AND RENATO,1,2 January 1983- 8 January 1983,1983-01-02,OLD,1,17.0,1983-02-12
1,2,YOU CAN'T HURRY LOVE,PHIL COLLINS,2,2 January 1983- 8 January 1983,1983-01-02,OLD,1,17.0,1983-03-19
2,3,A WINTER'S TALE,DAVID ESSEX,3,2 January 1983- 8 January 1983,1983-01-02,OLD,1,11.0,1983-02-12
3,4,BEST YEARS OF OUR LIVES,MODERN ROMANCE,4,2 January 1983- 8 January 1983,1983-01-02,OLD,1,14.0,1983-02-05
4,5,OUR HOUSE,MADNESS,5,2 January 1983- 8 January 1983,1983-01-02,OLD,1,14.0,1983-02-19
149375,5,OUR HOUSE,MADNESS,92,10 June 2012- 16 June 2012,2012-06-10,RE,2,1.0,2012-06-16
150382,5,OUR HOUSE,MADNESS,99,19 August 2012- 25 August 2012,2012-08-19,RE,3,1.0,2012-08-25
5,6,TIME (CLOCK OF THE HEART),CULTURE CLUB,6,2 January 1983- 8 January 1983,1983-01-02,OLD,1,13.0,1983-02-12
1492,6,TIME (CLOCK OF THE HEART),CULTURE CLUB,94,10 April 1983- 16 April 1983,1983-04-10,RE,2,1.0,1983-04-16
6,7,THE SHAKIN' STEVENS EP,SHAKIN' STEVENS,7,2 January 1983- 8 January 1983,1983-01-02,OLD,1,9.0,1983-01-29


In [48]:
# Identify the start date of the next chart run for each song

next_run_start = (chart_duration.groupby('song_id')['Run Start Date']
                  .shift(-1))

next_run_start.head(10)

0               NaT
1               NaT
2               NaT
3               NaT
4        2012-06-10
149375   2012-08-19
150382          NaT
5        1983-04-10
1492            NaT
6               NaT
Name: Run Start Date, dtype: datetime64[us]

In [49]:
chart_duration['Next Run Start'] = next_run_start

chart_duration.head(10)

,song_id,Song,Artist,Position,Week,Run Start Date,Run Start,Run ID,Run Weeks Count,Run End Date,Next Run Start
0,1,SAVE YOUR LOVE,RENEE AND RENATO,1,2 January 1983- 8 January 1983,1983-01-02,OLD,1,17.0,1983-02-12,NaT
1,2,YOU CAN'T HURRY LOVE,PHIL COLLINS,2,2 January 1983- 8 January 1983,1983-01-02,OLD,1,17.0,1983-03-19,NaT
2,3,A WINTER'S TALE,DAVID ESSEX,3,2 January 1983- 8 January 1983,1983-01-02,OLD,1,11.0,1983-02-12,NaT
3,4,BEST YEARS OF OUR LIVES,MODERN ROMANCE,4,2 January 1983- 8 January 1983,1983-01-02,OLD,1,14.0,1983-02-05,NaT
4,5,OUR HOUSE,MADNESS,5,2 January 1983- 8 January 1983,1983-01-02,OLD,1,14.0,1983-02-19,2012-06-10
149375,5,OUR HOUSE,MADNESS,92,10 June 2012- 16 June 2012,2012-06-10,RE,2,1.0,2012-06-16,2012-08-19
150382,5,OUR HOUSE,MADNESS,99,19 August 2012- 25 August 2012,2012-08-19,RE,3,1.0,2012-08-25,NaT
5,6,TIME (CLOCK OF THE HEART),CULTURE CLUB,6,2 January 1983- 8 January 1983,1983-01-02,OLD,1,13.0,1983-02-12,1983-04-10
1492,6,TIME (CLOCK OF THE HEART),CULTURE CLUB,94,10 April 1983- 16 April 1983,1983-04-10,RE,2,1.0,1983-04-16,NaT
6,7,THE SHAKIN' STEVENS EP,SHAKIN' STEVENS,7,2 January 1983- 8 January 1983,1983-01-02,OLD,1,9.0,1983-01-29,NaT


### Calculate gaps between chart runs

The gap between two consecutive chart runs is measured in complete chart weeks.

This keeps the gap measure consistent with `Run Weeks Count`, which also represents chart duration in weeks.

In [50]:
chart_duration['Gap'] = (chart_duration['Next Run Start'] - 
                         chart_duration['Run End Date'])
chart_duration.head(10)

,song_id,Song,Artist,Position,Week,Run Start Date,Run Start,Run ID,Run Weeks Count,Run End Date,Next Run Start,Gap
0,1,SAVE YOUR LOVE,RENEE AND RENATO,1,2 January 1983- 8 January 1983,1983-01-02,OLD,1,17.0,1983-02-12,NaT,NaT
1,2,YOU CAN'T HURRY LOVE,PHIL COLLINS,2,2 January 1983- 8 January 1983,1983-01-02,OLD,1,17.0,1983-03-19,NaT,NaT
2,3,A WINTER'S TALE,DAVID ESSEX,3,2 January 1983- 8 January 1983,1983-01-02,OLD,1,11.0,1983-02-12,NaT,NaT
3,4,BEST YEARS OF OUR LIVES,MODERN ROMANCE,4,2 January 1983- 8 January 1983,1983-01-02,OLD,1,14.0,1983-02-05,NaT,NaT
4,5,OUR HOUSE,MADNESS,5,2 January 1983- 8 January 1983,1983-01-02,OLD,1,14.0,1983-02-19,2012-06-10,10704 days
149375,5,OUR HOUSE,MADNESS,92,10 June 2012- 16 June 2012,2012-06-10,RE,2,1.0,2012-06-16,2012-08-19,64 days
150382,5,OUR HOUSE,MADNESS,99,19 August 2012- 25 August 2012,2012-08-19,RE,3,1.0,2012-08-25,NaT,NaT
5,6,TIME (CLOCK OF THE HEART),CULTURE CLUB,6,2 January 1983- 8 January 1983,1983-01-02,OLD,1,13.0,1983-02-12,1983-04-10,57 days
1492,6,TIME (CLOCK OF THE HEART),CULTURE CLUB,94,10 April 1983- 16 April 1983,1983-04-10,RE,2,1.0,1983-04-16,NaT,NaT
6,7,THE SHAKIN' STEVENS EP,SHAKIN' STEVENS,7,2 January 1983- 8 January 1983,1983-01-02,OLD,1,9.0,1983-01-29,NaT,NaT


### Convert gap durations to complete chart weeks

The gap is initially calculated as a `Timedelta`, representing the exact difference
between the end of one chart run and the start of the next.

For the gap analysis, this duration is converted to complete weeks by dividing the
number of days by 7 using floor division.

This means that only complete chart weeks are counted.

In [51]:
chart_duration['Gap Weeks'] = chart_duration['Gap'].dt.days // 7
chart_duration.head(10)

,song_id,Song,Artist,Position,Week,Run Start Date,Run Start,Run ID,Run Weeks Count,Run End Date,Next Run Start,Gap,Gap Weeks
0,1,SAVE YOUR LOVE,RENEE AND RENATO,1,2 January 1983- 8 January 1983,1983-01-02,OLD,1,17.0,1983-02-12,NaT,NaT,NaN
1,2,YOU CAN'T HURRY LOVE,PHIL COLLINS,2,2 January 1983- 8 January 1983,1983-01-02,OLD,1,17.0,1983-03-19,NaT,NaT,NaN
2,3,A WINTER'S TALE,DAVID ESSEX,3,2 January 1983- 8 January 1983,1983-01-02,OLD,1,11.0,1983-02-12,NaT,NaT,NaN
3,4,BEST YEARS OF OUR LIVES,MODERN ROMANCE,4,2 January 1983- 8 January 1983,1983-01-02,OLD,1,14.0,1983-02-05,NaT,NaT,NaN
4,5,OUR HOUSE,MADNESS,5,2 January 1983- 8 January 1983,1983-01-02,OLD,1,14.0,1983-02-19,2012-06-10,10704 days,1529.0
149375,5,OUR HOUSE,MADNESS,92,10 June 2012- 16 June 2012,2012-06-10,RE,2,1.0,2012-06-16,2012-08-19,64 days,9.0
150382,5,OUR HOUSE,MADNESS,99,19 August 2012- 25 August 2012,2012-08-19,RE,3,1.0,2012-08-25,NaT,NaT,NaN
5,6,TIME (CLOCK OF THE HEART),CULTURE CLUB,6,2 January 1983- 8 January 1983,1983-01-02,OLD,1,13.0,1983-02-12,1983-04-10,57 days,8.0
1492,6,TIME (CLOCK OF THE HEART),CULTURE CLUB,94,10 April 1983- 16 April 1983,1983-04-10,RE,2,1.0,1983-04-16,NaT,NaT,NaN
6,7,THE SHAKIN' STEVENS EP,SHAKIN' STEVENS,7,2 January 1983- 8 January 1983,1983-01-02,OLD,1,9.0,1983-01-29,NaT,NaT,NaN


In [52]:
chart_duration.info()

<class 'pandas.DataFrame'>
Index: 47056 entries, 0 to 210879
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype          
---  ------           --------------  -----          
 0   song_id          47056 non-null  int64          
 1   Song             47056 non-null  str            
 2   Artist           47056 non-null  str            
 3   Position         47056 non-null  int64          
 4   Week             47056 non-null  str            
 5   Run Start Date   47056 non-null  datetime64[us] 
 6   Run Start        47056 non-null  str            
 7   Run ID           47056 non-null  int64          
 8   Run Weeks Count  47056 non-null  float64        
 9   Run End Date     47056 non-null  datetime64[us] 
 10  Next Run Start   10404 non-null  datetime64[us] 
 11  Gap              10404 non-null  timedelta64[us]
 12  Gap Weeks        10404 non-null  float64        
dtypes: datetime64[us](3), float64(2), int64(3), str(4), timedelta64[us](1)
memory usage: 5.0 MB

#### Check for duplicate chart runs

The combination of `song_id` and `Run ID` should uniquely identify each run.
The six duplicates correspond to the records with missing artist information.

In [53]:
duplicates = chart_duration.duplicated(['song_id', 'Run ID']).sum()

print(f'Duplicate Run Entries: {duplicates}')

Duplicate Run Entries: 0


#### Check for negative gaps

A negative gap indicates that the next chart run starts before the previous
run has ended. This represents an overlap in the calculated chart history.

Since the analysis focuses on the Top 50 songs, only runs with a chart position
of 50 or better are examined here.

In [54]:
# Check runs with negative gaps

negative_gaps = (chart_duration['Gap Weeks'] < 0) & (
                chart_duration['Position'] <= 50)

print(f'Gap Weeks with negative gaps: {len(chart_duration.loc[negative_gaps,:])}')

chart_duration.loc[negative_gaps, :]

Gap Weeks with negative gaps: 9


,song_id,Song,Artist,Position,Week,Run Start Date,Run Start,Run ID,Run Weeks Count,Run End Date,Next Run Start,Gap,Gap Weeks
44399,8801,INSANITY,OCEANIC,21,18 August 1991- 24 August 1991,1991-08-18,OLD,1,5.0,1991-08-24,1991-08-18,-6 days,-1.0
45451,9059,YOU SHOWED ME,SALT-N-PEPA,22,24 November 1991- 30 November 1991,1991-11-24,OLD,1,3.0,1991-11-30,1991-11-24,-6 days,-1.0
45823,9153,THE DEVIL YOU KNOW,JESUS JONES,19,3 January 1993- 9 January 1993,1993-01-03,OLD,1,6.0,1993-01-09,1993-01-03,-6 days,-1.0
50029,10317,SWEET THING,MICK JAGGER,24,31 January 1993- 6 February 1993,1993-01-31,OLD,1,3.0,1993-02-06,1993-01-31,-6 days,-1.0
52505,11000,JEWEL,CRANES,29,19 September 1993- 25 September 1993,1993-09-19,OLD,1,7.0,1993-09-25,1993-09-19,-6 days,-1.0
52812,11077,DOWN IN A HOLE,ALICE IN CHAINS,36,17 October 1993- 23 October 1993,1993-10-17,OLD,1,3.0,1993-10-23,1993-10-17,-6 days,-1.0
61893,13494,COUNTRY HOUSE,BLUR,1,20 August 1995- 26 August 1995,1995-08-20,NEW,1,3.0,1995-09-09,1995-09-03,-6 days,-1.0
63498,13963,THE BEST THINGS IN LIFE A FE,LUTHER VANDROSS/JANET JACKSON,7,10 December 1995- 16 December 1995,1995-12-10,NEW,1,3.0,1995-12-30,1995-12-24,-6 days,-1.0
63500,13964,THE GIFT OF CHRISTMAS,CHILDLINERS,9,10 December 1995- 16 December 1995,1995-12-10,NEW,1,2.0,1995-12-23,1995-12-17,-6 days,-1.0


Negative gaps in the selected Top-50 data were inspected manually. 
The identified cases are caused by duplicate chart entries within the same chart week. 
These additional entries are treated as erroneous records and excluded from the gap analysis.

In [55]:
song_id = chart_duration.loc[:,'song_id'] == 13494
chart_duration.loc[song_id, :]

,song_id,Song,Artist,Position,Week,Run Start Date,Run Start,Run ID,Run Weeks Count,Run End Date,Next Run Start,Gap,Gap Weeks
61893,13494,COUNTRY HOUSE,BLUR,1,20 August 1995- 26 August 1995,1995-08-20,NEW,1,3.0,1995-09-09,1995-09-03,-6 days,-1.0
62148,13494,COUNTRY HOUSE,BLUR,57,3 September 1995- 9 September 1995,1995-09-03,NEW,2,10.0,1995-11-04,1995-12-31,57 days,8.0
63888,13494,COUNTRY HOUSE,BLUR,97,31 December 1995- 6 January 1996,1995-12-31,RE,3,1.0,1996-01-06,NaT,NaT,NaN


In [56]:
# Explore the original chart history of an overlapping run

song_id = chart_history.loc[:,'song_id'] == 13494
chart_history.loc[song_id, :]

,song_id,Song,Artist,Position,Last Week,Peak,Weeks on Chart,Week
61893,13494,COUNTRY HOUSE,BLUR,1,LW:New,1,1,20 August 1995- 26 August 1995
61993,13494,COUNTRY HOUSE,BLUR,1,LW:1,1,2,27 August 1995- 2 September 1995
62093,13494,COUNTRY HOUSE,BLUR,2,LW:1,1,3,3 September 1995- 9 September 1995
62148,13494,COUNTRY HOUSE,BLUR,57,LW:New,57,1,3 September 1995- 9 September 1995
62195,13494,COUNTRY HOUSE,BLUR,4,LW:2,1,4,10 September 1995- 16 September 1995
62267,13494,COUNTRY HOUSE,BLUR,76,LW:57,57,2,10 September 1995- 16 September 1995
62302,13494,COUNTRY HOUSE,BLUR,11,LW:4,1,5,17 September 1995- 23 September 1995
62410,13494,COUNTRY HOUSE,BLUR,19,LW:11,1,6,24 September 1995- 30 September 1995
62517,13494,COUNTRY HOUSE,BLUR,26,LW:19,1,7,1 October 1995- 7 October 1995
62621,13494,COUNTRY HOUSE,BLUR,30,LW:26,1,8,8 October 1995- 14 October 1995


## Data quality correction

Duplicate chart entries occurring within the same chart week were identified among the Top-50 songs. 

These entries create artificial overlapping chart runs and are excluded from the analysis.

In [57]:
songs_to_delete = [{'song_id': 13494.0,
                    'Position': 57,
                    'Last Week': 'LW:New',
                    'Week': '3 September 1995- 9 September 1995',
                    'Run Start': 'NEW'},
                    {'song_id': 13494.0,
                     'Position': 76,
                     'Last Week': 'LW:57',
                     'Week': '10 September 1995- 16 September 1995',
                     'Run Start': ''},
                    {'song_id': 13963.0,
                     'Position': 91,
                     'Last Week': 'LW:New',
                     'Week': '24 December 1995- 30 December 1995',
                     'Run Start': 'NEW'},
                    {'song_id': 13964.0,
                     'Position': 61,
                     'Last Week': 'LW:New',
                     'Week': '17 December 1995- 23 December 1995',
                     'Run Start': 'NEW'},
                    {'song_id': 9153.0,
                     'Position': 19,
                     'Last Week': 'LW:New',
                     'Week': '3 January 1993- 9 January 1993',
                     'Run Start': 'NEW'},
                    {'song_id': 8801.0,
                     'Position': 21,
                     'Last Week': 'LW:New',
                     'Week': '18 August 1991- 24 August 1991',
                     'Run Start': 'NEW'},
                    {'song_id': 9059.0,
                     'Position': 22,
                     'Last Week': 'LW:New',
                     'Week': '24 November 1991- 30 November 1991',
                     'Run Start': 'NEW'},
                    {'song_id': 10317.0,
                     'Position': 24,
                     'Last Week': 'LW:New',
                     'Week': '31 January 1993- 6 February 1993',
                     'Run Start': 'NEW'},
                    {'song_id': 11077.0,
                     'Position': 36,
                     'Last Week': 'LW:New',
                     'Week': '17 October 1993- 23 October 1993',
                     'Run Start': 'NEW'},
                    {'song_id': 11000.0,
                     'Position': 29,
                     'Last Week': 'LW:New',
                     'Week': '19 September 1993- 25 September 1993',
                     'Run Start': 'NEW'}]

# Sources:
# https://www.officialcharts.com/charts/singles-chart/19950903/7501/
# https://www.officialcharts.com/charts/singles-chart/19950910/7501/
# https://www.officialcharts.com/charts/singles-chart/19951224/7501/
# https://www.officialcharts.com/charts/singles-chart/19951217/7501/
# https://www.officialcharts.com/charts/singles-chart/19920103/7501/
# https://www.officialcharts.com/charts/singles-chart/19910818/7501/
# https://www.officialcharts.com/charts/singles-chart/19911124/7501/
# https://www.officialcharts.com/charts/singles-chart/19930131/7501/
# https://www.officialcharts.com/charts/singles-chart/19931017/7501/
# https://www.officialcharts.com/charts/singles-chart/19930919/7501/

affected_song_ids = {item['song_id'] for item in songs_to_delete}

print(f'Number of affected songs: {len(affected_song_ids)}')

Number of affected songs: 9


In [58]:
print(f'Chart duration rows before correction: {len(chart_duration)}')

Chart duration rows before correction: 47056


In [59]:
for item in songs_to_delete:
    mask = ((chart_duration['song_id'] == item['song_id']) &
            (chart_duration['Position'] == item['Position']) &
            (chart_duration['Week'] == item['Week']) &
            (chart_duration['Run Start'] == item['Run Start']))

    chart_duration = chart_duration.loc[~mask].copy()

In [60]:
print(f'Chart duration rows after correction: {len(chart_duration)}')

Chart duration rows after correction: 47047


In [61]:
# Check runs with negative gaps

negative_gaps = (chart_duration['Gap Weeks'] < 0) & (
                chart_duration['Position'] <= 50)

print(f'Gap Weeks with negative gaps: {len(chart_duration.loc[negative_gaps,:])}')

chart_duration.loc[negative_gaps, :]

Gap Weeks with negative gaps: 9


,song_id,Song,Artist,Position,Week,Run Start Date,Run Start,Run ID,Run Weeks Count,Run End Date,Next Run Start,Gap,Gap Weeks
44399,8801,INSANITY,OCEANIC,21,18 August 1991- 24 August 1991,1991-08-18,OLD,1,5.0,1991-08-24,1991-08-18,-6 days,-1.0
45451,9059,YOU SHOWED ME,SALT-N-PEPA,22,24 November 1991- 30 November 1991,1991-11-24,OLD,1,3.0,1991-11-30,1991-11-24,-6 days,-1.0
45823,9153,THE DEVIL YOU KNOW,JESUS JONES,19,3 January 1993- 9 January 1993,1993-01-03,OLD,1,6.0,1993-01-09,1993-01-03,-6 days,-1.0
50029,10317,SWEET THING,MICK JAGGER,24,31 January 1993- 6 February 1993,1993-01-31,OLD,1,3.0,1993-02-06,1993-01-31,-6 days,-1.0
52505,11000,JEWEL,CRANES,29,19 September 1993- 25 September 1993,1993-09-19,OLD,1,7.0,1993-09-25,1993-09-19,-6 days,-1.0
52812,11077,DOWN IN A HOLE,ALICE IN CHAINS,36,17 October 1993- 23 October 1993,1993-10-17,OLD,1,3.0,1993-10-23,1993-10-17,-6 days,-1.0
61893,13494,COUNTRY HOUSE,BLUR,1,20 August 1995- 26 August 1995,1995-08-20,NEW,1,3.0,1995-09-09,1995-09-03,-6 days,-1.0
63498,13963,THE BEST THINGS IN LIFE A FE,LUTHER VANDROSS/JANET JACKSON,7,10 December 1995- 16 December 1995,1995-12-10,NEW,1,3.0,1995-12-30,1995-12-24,-6 days,-1.0
63500,13964,THE GIFT OF CHRISTMAS,CHILDLINERS,9,10 December 1995- 16 December 1995,1995-12-10,NEW,1,2.0,1995-12-23,1995-12-17,-6 days,-1.0


#### Recalculate gaps after data correction

The identified duplicate run entries have been removed from `chart_duration`.

As `Next Run Start` depends on the sequence of chart runs, it is recalculated after the correction. 
The gap duration is then recalculated in complete chart weeks.

In [62]:
# Recalculate the start date of the next chart run

next_run_start = (chart_duration.groupby('song_id')['Run Start Date']
                  .shift(-1))

chart_duration['Next Run Start'] = next_run_start

In [63]:
chart_duration['Gap'] = (chart_duration['Next Run Start'] -
                         chart_duration['Run End Date'])

chart_duration['Gap Weeks'] = chart_duration['Gap'].dt.days // 7

In [64]:
# Check runs with negative gaps

negative_gaps = (chart_duration['Gap Weeks'] < 0) & (
                chart_duration['Position'] <= 50)

print(f'Gap Weeks with negative gaps: {len(chart_duration.loc[negative_gaps,:])}')

chart_duration.loc[negative_gaps, :]

Gap Weeks with negative gaps: 0


,song_id,Song,Artist,Position,Week,Run Start Date,Run Start,Run ID,Run Weeks Count,Run End Date,Next Run Start,Gap,Gap Weeks


In [65]:
chart_duration.loc[chart_duration['song_id'] == 13494, :]

,song_id,Song,Artist,Position,Week,Run Start Date,Run Start,Run ID,Run Weeks Count,Run End Date,Next Run Start,Gap,Gap Weeks
61893,13494,COUNTRY HOUSE,BLUR,1,20 August 1995- 26 August 1995,1995-08-20,NEW,1,3.0,1995-09-09,1995-12-31,113 days,16.0
63888,13494,COUNTRY HOUSE,BLUR,97,31 December 1995- 6 January 1996,1995-12-31,RE,3,1.0,1996-01-06,NaT,NaT,NaN


In [66]:
# Remove intermediate columns

chart_duration = chart_duration.drop(columns=['Week', 'Next Run Start', 'Gap'])

chart_duration.head()

,song_id,Song,Artist,Position,Run Start Date,Run Start,Run ID,Run Weeks Count,Run End Date,Gap Weeks
0,1,SAVE YOUR LOVE,RENEE AND RENATO,1,1983-01-02,OLD,1,17.0,1983-02-12,NaN
1,2,YOU CAN'T HURRY LOVE,PHIL COLLINS,2,1983-01-02,OLD,1,17.0,1983-03-19,NaN
2,3,A WINTER'S TALE,DAVID ESSEX,3,1983-01-02,OLD,1,11.0,1983-02-12,NaN
3,4,BEST YEARS OF OUR LIVES,MODERN ROMANCE,4,1983-01-02,OLD,1,14.0,1983-02-05,NaN
4,5,OUR HOUSE,MADNESS,5,1983-01-02,OLD,1,14.0,1983-02-19,1529.0


In [67]:
chart_duration.info()

<class 'pandas.DataFrame'>
Index: 47047 entries, 0 to 210879
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   song_id          47047 non-null  int64         
 1   Song             47047 non-null  str           
 2   Artist           47047 non-null  str           
 3   Position         47047 non-null  int64         
 4   Run Start Date   47047 non-null  datetime64[us]
 5   Run Start        47047 non-null  str           
 6   Run ID           47047 non-null  int64         
 7   Run Weeks Count  47047 non-null  float64       
 8   Run End Date     47047 non-null  datetime64[us]
 9   Gap Weeks        10395 non-null  float64       
dtypes: datetime64[us](2), float64(2), int64(3), str(3)
memory usage: 3.9 MB


## Data quality correction

Duplicate chart entries occurring within the same chart week were identified
among the Top-50 songs.

These entries create artificial overlapping chart runs and are excluded from
the analysis.

The identified records are removed from the original chart history before
the final analysis datasets are created.

In [68]:
print(f'Chart history rows before correction: {len(chart_history)}')

Chart history rows before correction: 210882


In [69]:
# Data quality correction

for item in songs_to_delete:
    mask = ((chart_history['song_id'] == item['song_id']) &
            (chart_history['Position'] == item['Position']) &
            (chart_history['Last Week'] == item['Last Week']) &
            (chart_history['Week'] == item['Week']))

    chart_history = chart_history.loc[~mask].copy()

In [70]:
print(f'Chart history rows after correction: {len(chart_history)}')

Chart history rows after correction: 210872


In [71]:
chart_history.info()

<class 'pandas.DataFrame'>
Index: 210872 entries, 0 to 210881
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   song_id         210872 non-null  int64
 1   Song            210872 non-null  str  
 2   Artist          210872 non-null  str  
 3   Position        210872 non-null  int64
 4   Last Week       210872 non-null  str  
 5   Peak            210872 non-null  int64
 6   Weeks on Chart  210872 non-null  int64
 7   Week            210872 non-null  str  
dtypes: int64(4), str(4)
memory usage: 14.5 MB


## Export Analysis Datasets

The datasets created during this notebook are exported for reuse in subsequent analyses.

The run-level dataset contains one row per chart run together with the derived
run duration and gap variables used for the gap analysis.

In [72]:
interim_path = project_path / 'data' / 'interim'

chart_duration.to_csv(interim_path/'chart_duration.csv', index=False)

chart_history.to_csv(interim_path/'uk_chart_history_clean.csv', index=False)
